In [ ]:
# 1
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None

    DEFAULT_MODELS = ['AMWP11.5-8200AC']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[0:1])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: AMWP11.5-8200AC 备件手册  英文 （SM052320111_Rev1.5）


,Item,Part No.,Description,Qty.,Section,Node
0,1,5168.0,Screw GB/T 818 M6×10 - H,26,Chassis,Chassis Covers Installation
1,2,10031293.0,Left cover,1,Chassis,Chassis Covers Installation
2,3,50000112.0,Nut QC/T 885 M6,4,Chassis,Chassis Covers Installation
3,4,10026016.0,Cover,2,Chassis,Chassis Covers Installation
4,5,8823.0,Screw GB/T 818 M6×8 - H,4,Chassis,Chassis Covers Installation


100%|██████████| 1/1 [00:02<00:00,  2.84s/it]


: 

In [ ]:
# 2
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['BA16NE', 'BA18NE']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            return "; ".join(DEFAULT_MODELS)
        
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[1:2])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA16  18NE  Parts Manual （SM042220111_Rev1.1）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1.0,NaN,Left Hub Assembly (Refer to 1-2),1,NaN,Chassis,Steer Cylinder Assembly Installations
1,2.0,50000921.0,"Pin, Pivot",4,NaN,Chassis,Steer Cylinder Assembly Installations
2,3.0,50000926.0,Tie-Rod,2,NaN,Chassis,Steer Cylinder Assembly Installations
3,4.0,1257.0,Nut GB/T 6182 M16,4,NaN,Chassis,Steer Cylinder Assembly Installations
4,5.0,997.0,Flat Washer GB/T 97.1 16,4,NaN,Chassis,Steer Cylinder Assembly Installations


100%|██████████| 1/1 [00:00<00:00,  1.08it/s]


In [60]:
# 3
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['BA20ERT', 'BA22ERT']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            return "; ".join(DEFAULT_MODELS)
        
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[2:3])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA20ERT BA22ERT Parts Manual（SM041920111_Rev1.1）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,2452.0,Bolt GB/T 5783 M12 × 45,8,NaN,Chassis,Axle Installations
1,2,814.0,Flat Washer GB/T 97.1 12,24,NaN,Chassis,Axle Installations
2,3,50005351.0,Bracket,2,NaN,Chassis,Axle Installations
3,4,50006997.0,Bracket (Left Side),1,NaN,Chassis,Axle Installations
4,5,2059.0,Bolt GB/T 5783 M12 × 30,8,NaN,Chassis,Axle Installations


100%|██████████| 1/1 [00:01<00:00,  1.09s/it]


In [56]:
# 4
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['BA16CERT2', 'BA18CERT2', 'BA20CERT2', 'BA22CERT2']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            return "; ".join(DEFAULT_MODELS)
        
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[3:4])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA22 20 18 16 CERT2 备件手册  英文（SM042320113_Rev1.4）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,10633.0,Rear Axle Assembly,1.0,NaN,Chassis,Axle Installations (2WS)
1,2,2904.0,Bolt GB/T 5783 M14 × 40,8.0,NaN,Chassis,Axle Installations (2WS)
2,3,1258.0,Spring Washer GB/T 93 14,8.0,NaN,Chassis,Axle Installations (2WS)
3,4,1043.0,Flat Washer GB/T 97.1 14,8.0,NaN,Chassis,Axle Installations (2WS)
4,5,4487.0,Nut GB/T 6182 M22,16.0,NaN,Chassis,Axle Installations (2WS)


100%|██████████| 1/1 [00:01<00:00,  1.23s/it]


In [46]:
# 5
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['BA16CRT', 'BA18CRT', 'BA20CRT', 'BA22CRT']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            return "; ".join(DEFAULT_MODELS)
        
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[4:5])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA22 20 18 16 CRT Parts Manual （SM042220115_Rev2.0）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,9646.0,Rear Axle Assembly,1,NaN,Chassis,Axle Installations (2WS)
1,2,4487.0,Nut GB/T 6182 M22,16,NaN,Chassis,Axle Installations (2WS)
2,3,2550.0,Flat Washer GB/T 97.1 22,32,NaN,Chassis,Axle Installations (2WS)
3,4,92046502.0,Drive Motor Assembly,1,NaN,Chassis,Axle Installations (2WS)
4,5,1055.0,Bolt GB/T 5783 M12 × 40,4,NaN,Chassis,Axle Installations (2WS)


100%|██████████| 1/1 [00:01<00:00,  1.29s/it]


00001251

In [45]:
# 6
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['BA16CRT2', 'BA18CRT2', 'BA20CRT2', 'BA22CRT2']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[5:6])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA22 20 18 16 CRT2 备件手册  英文 （SM042320111_Rev1.5）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,9646.0,Rear Axle Assembly,1,NaN,Chassis,Axle Installations (2WS)
1,2,4487.0,Nut GB/T 6182 M22,16,NaN,Chassis,Axle Installations (2WS)
2,3,2550.0,Flat Washer GB/T 97.1 22,32,NaN,Chassis,Axle Installations (2WS)
3,4,92046502.0,Drive Motor Assembly,1,NaN,Chassis,Axle Installations (2WS)
4,5,1055.0,Bolt GB/T 5783 M12 × 40,4,NaN,Chassis,Axle Installations (2WS)


100%|██████████| 1/1 [00:01<00:00,  1.02s/it]


50006515


In [1]:
# 7
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['BA24RT', 'BA28RT', 'BA24BRT', 'BA28BRT']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[6:7])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

C:\Users\Normc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4
  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA24  28 RT BRT  Parts Manual（SM041920113_Rev2.0）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,92100196.0,Rear Axle Assembly,1.0,NaN,Chassis,Axle Installations
1,2,4487.0,Nut GB/T 6182 M22,16.0,NaN,Chassis,Axle Installations
2,3,2550.0,Flat Washer GB/T 97.1 22,32.0,NaN,Chassis,Axle Installations
3,4,92050965.0,O-Ring,1.0,NaN,Chassis,Axle Installations
4,5,92046506.0,Drive Motor Assembly,1.0,NaN,Chassis,Axle Installations


100%|██████████| 1/1 [00:04<00:00,  4.60s/it]


In [2]:
# 8
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['BA36RT', 'BA41RT', 'BA44RT']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[7:8])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BA44  41  36RT  备件手册 英文（SM042420111_Rev1.2）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,678.0,Bolt GB/T 5783 M10 × 25,4,NaN,Chassis,Support Arm Installations (Left Front and Righ...
1,2,93000016.0,Spring Washer Ф10.2,4,NaN,Chassis,Support Arm Installations (Left Front and Righ...
2,3,219.0,"8902-10060 Pin,Lock",4,NaN,Chassis,Support Arm Installations (Left Front and Righ...
3,4,93301095.0,Pin,2,NaN,Chassis,Support Arm Installations (Left Front and Righ...
4,5,93000595.0,"Washer, Thrust",8,NaN,Chassis,Support Arm Installations (Left Front and Righ...


100%|██████████| 1/1 [00:10<00:00, 10.49s/it]


In [3]:
# 9
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = [
    "BT30ERT", "BT28ERT", "BT26ERT", "BT24ERT",
    "BT30SERT", "BT28SERT", "BT26SERT",
    "BT28BERT", "BT26BERT", "BT24BERT"
    ]

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[8:9])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BT24  26  28  30ERT SERT BERT  备件手册 英文（SM032020111_Rev2.2）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,92100488.0,Rear Axle Assembly \n(To SN BTEM023K00427),1.0,NaN,Chassis,Axle Installations (Standard)
1,1.B,92102524.0,Rear Axle Assembly \n(From SN BTEM023K00428),1.0,NaN,Chassis,Axle Installations (Standard)
2,2,4487.0,Nut GB/T 6182 M22,16.0,NaN,Chassis,Axle Installations (Standard)
3,3,2550.0,Flat Washer GB/T 97.1 22,32.0,NaN,Chassis,Axle Installations (Standard)
4,4,1249.0,Nut GB/T 6182 M12,8.0,NaN,Chassis,Axle Installations (Standard)


100%|██████████| 1/1 [00:07<00:00,  7.06s/it]


In [4]:
# 10
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = [
        "BT30RT", "BT28RT", "BT26RT", "BT24RT",
        "BT30SRT", "BT28SRT", "BT26SRT",
        "BT28BRT", "BT26BRT", "BT24BRT"
    ]


    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Remark") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[9:10])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: BT24  26  28  30RT SRT BRT  备件手册 英文（SM031920111_Rev2.2）


,Item,Part Number,Description,Qty.,Remark,Section,Node
0,1,92102526.0,Rear Axle Assembly,1.0,NaN,Chassis,Axle Installations
1,2,4487.0,Nut GB/T 6182 M22,16.0,NaN,Chassis,Axle Installations
2,3,2550.0,Flat Washer GB/T 97.1 22,32.0,NaN,Chassis,Axle Installations
3,4,92050965.0,O-Ring,1.0,NaN,Chassis,Axle Installations
4,5,92046506.0,Drive Motor Assembly,1.0,NaN,Chassis,Axle Installations


100%|██████████| 1/1 [00:06<00:00,  6.33s/it]


In [6]:
# 11
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['JCPT0608DCS', 'JCPT0708DCS']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[10:11])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT 0608 0708 DCS (SM0117225_Rev4.1) Delta


,Item,Part No.,Description,Qty.,Section,Node
0,1,10001396.0,Steer cylinder assembly (without seal kit),1,Chassis,Steer Linkage and Wheels Assembly
1,--,3068.0,Seal kit,1,Chassis,Steer Linkage and Wheels Assembly
2,2,1290.0,Straight fitting,2,Chassis,Steer Linkage and Wheels Assembly
3,3,10001075.0,Cover,2,Chassis,Steer Linkage and Wheels Assembly
4,4,10001073.0,Screw,2,Chassis,Steer Linkage and Wheels Assembly


100%|██████████| 1/1 [00:00<00:00,  1.75it/s]


In [7]:
# 12
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = ['JCPT0708DCS']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[11:12])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT0708DCS (SM0117225_Rev2.4)


,Item,Part No.,Description,Qty.,Section,Node
0,1,10001396,Steer cylinder assembly (without seal kit),1,Chassis,Steer Linkage and Wheels Assembly
1,--,3068,Seal kit,1,Chassis,Steer Linkage and Wheels Assembly
2,2,1290,Straight fitting,2,Chassis,Steer Linkage and Wheels Assembly
3,3,10001075,Cover,2,Chassis,Steer Linkage and Wheels Assembly
4,4,10001073,Screw,2,Chassis,Steer Linkage and Wheels Assembly


100%|██████████| 1/1 [00:01<00:00,  1.18s/it]


In [13]:
# 13
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = [
    "JCPT0808",
    "JCPT0812",
    "JCPT1008",
    "JCPT1012",
    "JCPT1212",
    "JCPT1412",
    "JCPT1612",
    "JCPT1614"
    ]

    MODEL_RE = re.compile(r"JCPT\d{4}", re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[12:13])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT0808-1612AC HA Parts Manual（SM012020111_Rev1.6）


,Item,Part No.,Description,Qty.,Section,Node
0,1,10000482.0,Cover,1,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
1,2,10000596.0,Screw,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
2,3,1029.0,Nut GB/T 6170 M12,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
3,4,1252.0,Screw GB/T 70.1 M5×12,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
4,5,10000482.0,Cover,1,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...


100%|██████████| 1/1 [00:00<00:00,  1.66it/s]


In [14]:
# 14
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    DEFAULT_MODELS = [
        "JCPT0808", "JCPT0812", "JCPT1008", "JCPT1012",
        "JCPT1212", "JCPT1412", "JCPT1612", "JCPT1614"
    ]


    MODEL_RE = re.compile(r"JCPT\d{4}", re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[13:14])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT0808-1612HA AC Parts Manual（SM012020111_Rev1.6）


,Item,Part No.,Description,Qty.,Section,Node
0,1,10000482.0,Cover,1,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
1,2,10000596.0,Screw,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
2,3,1029.0,Nut GB/T 6170 M12,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
3,4,1252.0,Screw GB/T 70.1 M5×12,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
4,5,10000482.0,Cover,1,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...


100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


In [15]:
# 15
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    # Дефолтные модели можно оставить в таком формате
    DEFAULT_MODELS = ["JCPT0808", "JCPT0812", "JCPT1008", "JCPT1012",
                    "JCPT1212", "JCPT1412", "JCPT1612", "JCPT1614"]

    # Регулярка ищет только JCPT + 4 цифры
    MODEL_RE = re.compile(r"JCPT\d{4}", re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[14:15])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT0808-1614 Магни  备件手册  英文（SM0117217_Rev1.0）


,Item,Part No.,Description,Qty.,Section,Node
0,1,10000596.0,Screw,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
1,2,1029.0,Nut GB/T 6170 M12,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
2,3,1030.0,Screw GB/T 70.1 M5×10,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
3,4,10000482.0,Cover,2,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...
4,5,429.0,8813-556550 Bearing,4,Chassis,Steer Linkage and Wheels Assembly (Hydraulic M...


100%|██████████| 1/1 [00:03<00:00,  3.32s/it]


In [16]:
# 16
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    # Дефолтные модели можно оставить в таком формате
    DEFAULT_MODELS = ['JCPT1218RT', 'JCPT1418RT']

    # Регулярка ищет только JCPT + 4 цифры
    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[15:16])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT1218RT   1418RT 备件手册  英文（SM0118211_Rev1.5）


,Item,Part No.,Description,Qty.,Section,Node
0,1,10002491.0,Frame weldment,1,Chassis,Steer Axle Assembly Installation
1,2,3611.0,895-38.1×355 Pin,1,Chassis,Steer Axle Assembly Installation
2,3,220.0,8902-10080 Pin,1,Chassis,Steer Axle Assembly Installation
3,4,679.0,Washer GB/T 93 10,1,Chassis,Steer Axle Assembly Installation
4,5,615.0,Washer GB/T 97.1 10,1,Chassis,Steer Axle Assembly Installation


100%|██████████| 1/1 [00:03<00:00,  3.36s/it]


In [17]:
# 17
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None


    # Дефолтные модели можно оставить в таком формате
    DEFAULT_MODELS = ['JCPT1523RTB', 'JCPT1823RTB']

    # Регулярка ищет только JCPT + 4 цифры
    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[16:17])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT1523RTB 1823RTB-SM012120119_Rev1.1-Parts Manual


,Item,Part No.,Description,Qty.,Section,Node
0,1,10019069.0,Frame weldment,1,Chassis,Steer Axle Assembly Installation
1,2,10004333.0,Oscillate cylinder,2,Chassis,Steer Axle Assembly Installation
2,--,3373.0,Counterbalance valve spool,1,Chassis,Steer Axle Assembly Installation
3,--,5325.0,Seal kit,1,Chassis,Steer Axle Assembly Installation
4,3,1291.0,Straight fitting,4,Chassis,Steer Axle Assembly Installation


100%|██████████| 1/1 [00:06<00:00,  6.29s/it]


In [18]:
# 18
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None

    DEFAULT_MODELS = ['JCPT1523RTL', 'JCPT1823RTL']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[17:18])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT1523RTL 1823RTL 备件手册 英文 （SM012220129_Rev1.2）


,Item,Part No.,Description,Qty.,Section,Node
0,1,10025895.0,Frame weldment,1,Chassis,Font Axle and Rear Axle Installation
1,2,438.0,8813-38.1×44.45×31.75 Bearing,4,Chassis,Font Axle and Rear Axle Installation
2,3,NaN,Left oscillate cylinder assembly (refer to 404.1),2,Chassis,Font Axle and Rear Axle Installation
3,4,7832.0,8812-404445 Bearing,2,Chassis,Font Axle and Rear Axle Installation
4,5,220.0,8902-10080 Pin,4,Chassis,Font Axle and Rear Axle Installation


100%|██████████| 1/1 [00:03<00:00,  3.39s/it]


In [20]:
# 19
if __name__ == '__main__':
    import os
    import camelot
    import pandas as pd
    from tqdm import tqdm
    from IPython.display import display
    import warnings
    from cryptography.utils import CryptographyDeprecationWarning
    import re

    warnings.filterwarnings("ignore")
    warnings.filterwarnings("ignore", category=CryptographyDeprecationWarning)

    folder = os.getcwd()
    pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]


    def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="all", title_height=120):
        import pdfplumber
        os.makedirs(out_dir, exist_ok=True)

        tables_lattice = camelot.read_pdf(pdf_path, pages=pages, flavor="lattice")

        with pdfplumber.open(pdf_path) as pdf:
            for i, t in enumerate(tables_lattice):
                t.to_csv(os.path.join(out_dir, f"lattice_{i}.csv"), index=False)

                page_num = int(t.page) - 1
                page = pdf.pages[page_num]

                x1, y1, x2, y2 = t._bbox
                page_w = page.width
                page_h = page.height

                table_top = page_h - y2

                left   = x1
                top    = max(0, table_top - title_height)
                right  = x2
                bottom = table_top

                left = max(0, min(left, page_w))
                right = max(0, min(right, page_w))
                top = max(0, min(top, page_h))
                bottom = max(0, min(bottom, page_h))

                if right <= left or bottom <= top:
                    title_text = ""
                else:
                    title_box = (left, top, right, bottom)
                    title_text = page.crop(title_box, strict=False).extract_text(
                        x_tolerance=2, y_tolerance=2
                    ) or ""

                title_text = title_text.strip()

                with open(os.path.join(out_dir, f"text_{i}.txt"), "w", encoding="utf-8") as f:
                    f.write(title_text)

        return tables_lattice


    def clean_title(title_text: str) -> str:
        lines = [l.strip() for l in title_text.splitlines() if l.strip()]
        s = " ".join(lines)
        s = re.sub(r"\bSection\s+\d+(\.\d+)?\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\bParts\s+Manual\b", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s+", " ", s).strip()
        return s


    ONLY_DIGITS_SPACES = re.compile(r'^\s*\d+(?:\s+\d+)*\s*$')  
    DIGITS_HEAVY = re.compile(r'^\s*[\d\s,./:-]+\s*$')         

    def is_numeric_garbage(s: str) -> bool:
        s = (s or "").strip()
        if not s:
            return False
        if ONLY_DIGITS_SPACES.match(s):
            return True
        if DIGITS_HEAVY.match(s) and not re.search(r'[A-Za-zА-Яа-я]', s):
            return True
        return False


    def read_table_context(text_path: str):
        if not os.path.exists(text_path):
            return None, None

        with open(text_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()

        first_line = lines[0] if len(lines) > 0 else ""
        second_line = lines[1] if len(lines) > 1 else ""

        final_title = clean_title(first_line)

        s2 = (second_line or "").strip()
        if " " in s2:
            second_line_clean = s2.split(" ", 1)[1].strip()
        else:
            second_line_clean = ""

        tail_parts = []
        for s in lines[2:]:
            if not s.strip():
                break
            if is_numeric_garbage(s):
                break
            tail_parts.append(s.strip())

        if tail_parts:
            second_line_clean = (second_line_clean + " " if second_line_clean else "") + " ".join(tail_parts)

        return final_title, second_line_clean


    def join_uniques(series):
        vals = series.dropna().astype(str).str.strip()
        uniq = []
        for v in vals:
            for item in v.split():
                if item not in uniq or item == 'Olny':
                    uniq.append(item)
        return " ".join(uniq) if uniq else None


    def join_ordered(series):
        vals = series.dropna().astype(str).str.strip()
        vals = [v for v in vals if v]
        seen = set()
        result = []
        for v in vals:
            if v not in seen:
                seen.add(v)
                result.append(v)
        return "; ".join(result) if result else None

    DEFAULT_MODELS = ['JCPT1912DC', 'JCPT2212DC']

    MODEL_RE = re.compile(r"\b(?:{})\b".format("|".join(map(re.escape, DEFAULT_MODELS))), re.IGNORECASE)

    def extract_models(row):
        remark_text = str(row.get("Description") or "")
        models = MODEL_RE.findall(remark_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        node_text = str(row.get("Node") or "")
        models = MODEL_RE.findall(node_text)
        models = [m.upper() for m in models]
        if models:
            return "; ".join(models)
        
        return None
    
    def merge_models(df: pd.DataFrame) -> str:
        models_series = df["Model"].dropna().astype(str)
        if models_series.empty or df["Model"].isna().any():
            # Если есть хотя бы один None, возвращаем все дефолтные
            return "; ".join(DEFAULT_MODELS)
        
        # Собираем уникальные модели
        uniq = set()
        for val in models_series:
            for m in val.split(";"):
                uniq.add(m.strip())
        return "; ".join(sorted(uniq))


    def merge_parts(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()

        # --- функция для агрегирования моделей ---
        def merge_models(group: pd.DataFrame) -> str:
            if group["Model"].isna().any():
                # Если есть хотя бы один None → возвращаем все дефолтные
                return "; ".join(DEFAULT_MODELS)
            
            uniq = set()
            for val in group["Model"].dropna().astype(str):
                for m in val.split(";"):
                    uniq.add(m.strip())
            return "; ".join(sorted(uniq)) if uniq else None

        # --- агрегируем остальные колонки ---
        agg_map = {
            "Description": join_ordered,
            "Qty": "sum",
        }
        if "Section" in df.columns:
            agg_map["Section"] = join_ordered
        if "Node" in df.columns:
            agg_map["Node"] = join_ordered
        if "Remark" in df.columns:
            agg_map["Remark"] = join_uniques

        df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

        # --- агрегируем модели ---
        df_agg["Models"] = df.groupby("Part Number").apply(merge_models).values

        return df_agg


    for i, pth in enumerate(tqdm(pdf_files[18:19])):
        
        filename = os.path.basename(pth)          
        filename = os.path.splitext(filename)[0]
        print(f'Обработка файла: {filename}')

        # extract_tables_camelot(pth, filename, pages="all", title_height=800)

        dfs = []

        for file in os.listdir(filename):
            if not file.startswith("lattice_") or not file.lower().endswith(".csv"):
                continue

            path = os.path.join(filename, file)
            df_i = pd.read_csv(path)

            if 'Item' not in df_i.columns:
                continue

            if 'Part Number' not in df_i.columns and 'Part No.' not in df_i.columns and 'Item' in df_i.columns:
                df_i.columns.values[1] = "Part Number"
                df_i.columns.values[2] = "Description"
                df_i.columns.values[3] = "Qty."
            if 'Note' in df_i.columns:
                df_i.columns.values[4] = "Remark"


            idx = file.replace("lattice_", "").rsplit(".", 1)[0]
            text_path = os.path.join(filename, f"text_{idx}.txt")
            section_val, node_val = read_table_context(text_path)

            df_i["Section"] = section_val
            df_i["Node"] = node_val

            dfs.append(df_i)

        df_all = pd.concat(dfs, ignore_index=True)
        display(df_all.head())

        if 'Part Number' not in df_all.columns.values:
            df_all.columns.values[1] = "Part Number"
        df_all["Qty"] = pd.to_numeric(df_all["Qty."], errors="coerce")
        df_all = df_all[df_all["Qty"].notna()]

        drop_cols = [c for c in ["Qty.", "Item"] if c in df_all.columns]
        df_all = df_all.drop(columns=drop_cols)

        if "Remark" in df_all.columns:
            df_all["Remark"] = (
                df_all["Remark"]
                .astype(str)
                .str.replace("\n", " ", regex=False)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )
            df_all.loc[df_all["Remark"].str.lower().eq("nan"), "Remark"] = None

        df_all = df_all[df_all["Part Number"].notna()]
        df_all = df_all[df_all["Part Number"].astype(str).str.strip() != ""]
        df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

        part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

        df_letters = df_all[part_numeric.isna()].copy()
        df_numbers = df_all[part_numeric.notna()].copy()

        df_letters["Model"] = df_letters.apply(extract_models, axis=1)
        df_numbers["Model"] = df_numbers.apply(extract_models, axis=1)

        if not df_numbers.empty:
            df_numbers["Part Number"] = (
                part_numeric[part_numeric.notna()]
                .astype(int)
                .astype(str)
                .str.zfill(8)
            )
        df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
        df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

        df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)
        df_merged.to_excel(rf"{filename}_модели.xlsx", sheet_name="Parts", index=False)

  0%|          | 0/1 [00:00<?, ?it/s]

Обработка файла: JCPT1912DC  2212DC


,Item,Part No.,Description,Qty.,Section,Node
0,1,10011205.0,Cover,4,Chassis,Steer Linkage and Wheels Installation
1,2,1011.0,Screw GB/T 973 M5×12,16,Chassis,Steer Linkage and Wheels Installation
2,3,10014417.0,Pin,2,Chassis,Steer Linkage and Wheels Installation
3,4,816.0,Bolt GB/T 5783 M12×25,2,Chassis,Steer Linkage and Wheels Installation
4,5,814.0,Washer GB/T 97.1 12,8,Chassis,Steer Linkage and Wheels Installation


100%|██████████| 1/1 [00:00<00:00,  1.94it/s]
